## BASIC DATA

This notebook extracts, transforms and loads background basic data for the ARC researcher project.  
-  FoR Codes (2008/2028 versions, 2 and 4 digit)
-  OA Topics (amd hence Scopus codes) mapped to FoR codes
-  Australian eligible institutions  
-  ORCID background data

In [28]:
import pathlib
from collections import defaultdict
from pathlib import Path
import iteround
import urllib3
import json
import jsonlines
import glob
import re
import pandas as pd
import duckdb
import contextlib
from unidecode import unidecode
from nameparser import HumanName
from time import sleep
import pyarrow
from utils.pandas_setup import pandas_setup
from utils.chained_get import chained_get
pandas_setup()


### Code outside classes for python API in duckdb

In [29]:

def normalise_name(in_name: str=None) -> dict:
    in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
    in_name = in_name.title()
    name = HumanName(in_name)
    if name.title == 'Md.Shahriar':
        name.first = 'Shahriar'
        name.title = ''
        # print(f'{name = }')
    if name.title == 'Mahdi':
        name.first = 'Mahdi'
        name.title = ''
        # print(f'{name = }')
    # print(f'{name = }')
    return name.as_dict()

def tidy_role(n: str=None) -> str:
    n = n.strip()
    n = re.sub('_', '-', n)
    n = re.sub('r-D', 'r - D', n)
    n = re.sub(' -A', ' - A', n)
    n = re.sub('t-A', 't - A', n)
    return n

def extract_person(dd_in: dict=None) -> dict:
    first = chained_get(dd_in, ['given-names', 'value'], pd.NA)
    family = chained_get(dd_in, ['family-name', 'value'], pd.NA)
    full = f'{first} {family}'
    normal = str(normalise_name(full))
    return 


In [30]:
class SetUp:

    def __init__(self):
        self.open_db()
        return

    def open_db(self):
        """
        Opens and sets up the database connections and configurations.
        """
        self.db = duckdb.connect('/home/lc/m/working/arc.duckdb')
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/working/ERA.duckdb'")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_dec24/duckdb/authors.duckdb'")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/openalex_dec24/duckdb/institutions.duckdb'")
        self.db.sql("ATTACH IF NOT EXISTS '/home/lc/m/working/orcid.duckdb'")
        for table in ['oa_author_prospects', 'orcid.orcid23', 'orcid.orcid_small', 'orcid.orcid.names', 'orcid.summaries', 'orcid.works']:
            self.db.sql(f"DROP TABLE IF EXISTS {table}")
        self.db.sql("ATTACH IF NOT EXISTS ':memory:' AS memory")
        self.db.sql("""                
                SET memory_limit = '56GB';
                SET threads =6;
                SET preserve_insertion_order = false;                                        
                SET enable_progress_bar = true;
                SET temp_directory = '/home/lc/m/.tmp';
            """)

        with contextlib.suppress(Exception):
            self.db.create_function('normalise_name', 
                                        normalise_name, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )
            # self.db.create_function('extract_works', 
            #                             extract_works, 
            #                             return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
            #                         )
        self.db.sql("SHOW ALL TABLES").show()
        return

In [31]:
class FieldsOfResearch(SetUp):

    def __init__(self):
        super().__init__()
        return

    def load_original_for_tables(self):
        """
        Load 2008 and 2020 FOR tables, clean them, and align by names
        There are around 20 names that do not match            self.db.create_function('normalise_name', 
                                        normalise_name, 
                                        return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
                                        exception_handling='return_null',
                                        null_handling='special',
                                        side_effects=True
                                    )
        Match these by hand and generate the matching table (old -> new)
        Load the full matched alignment and place in the ERA and ARC databases  
        """
        self._load_old()
        self._load_new()
        # self._find_missalign()
        self._align()
        aligned = self.aligned # for duckdb
        self.db.sql("CREATE OR REPLACE TABLE arc.old_for_to_new_for AS SELECT * FROM aligned")
        self.db.sql("CREATE OR REPLACE TABLE ERA.old_for_to_new_for AS SELECT * FROM aligned")
        with pd.ExcelWriter('../data/BASIC_FOR_DATA/old_for_to_new_for.xlsx') as writer:
            self.aligned.to_excel(writer, sheet_name='aligned')
        return

    def _load_old(self):
        # load basic table of 2008 FoR codes
        old = pd.read_excel('../data/BASIC_FOR_DATA/FOR_2008.xlsx', dtype=str).dropna().reset_index(drop=True)
        dd = {'type_': 0, 'code_': 1, 'name_': 2}
        for k, v in dd.items():
            old[k] = [x.split(' ', maxsplit=2)[v] for x in old.FOR]
        old['name_'] = [x.title().replace(' And ', ' and ').strip() for x in old['name_']]
        old = old.astype(str).drop(columns='FOR')
        print(f'{old.shape = }\n{old.head()}')
        self.old = old
        return

    def _load_new(self):
        # load original table of 2020 FoR odes
        new = pd.read_excel('../data/BASIC_FOR_DATA/anzsrc2020_for.xlsx', dtype=str, skiprows=7, nrows=240, usecols=[0, 1, 2], sheet_name='Table 2')
        mask = [x != x for x in new['DIVISION']]
        new[mask] = new[mask].shift(-1, axis=1) # line up columns
        new = new.iloc[1:, 0:2].astype(str)
        new.columns = ['code', 'name']
        new.insert(0, 'type', ['DIVISION' if len(x) == 2 else 'GROUP' for x in new.code])
        new['name'] = [name.title().replace('And', 'and').strip() for name in new.name]
        print(f'{new.shape = }\n{new.head()}')
        self.new = new
        return

    def _align(self):
        # concatentate the old, new and hand-aligned old codes
        print(self.old.head())
        print(self.new.head())
        self.aligned = self.old.set_index('name_', drop=False).join(self.new.set_index('name', drop=False)).dropna().sort_values('code').reset_index(drop=True)
        self.aligned = self.aligned.iloc[:, [1,2,0,3,4,5]]
        print(f'{self.aligned.shape = }\n{self.aligned.head(self.aligned.shape[0])}')
        miss = pd.read_excel('../data/BASIC_FOR_DATA/match_old_new_for_SAVE.xlsx', dtype=str, sheet_name='miss')
        self.aligned = pd.concat([self.aligned, miss], axis=0).sort_values('code').reset_index(drop=True)
        for col in ['name_', 'code_', 'type_']:
            self.new.insert(0, col, pd.NA)
        print(f'{self.new.shape = }\n{self.new.head()}')
        self.aligned = pd.concat([self.aligned, miss, self.new], axis=0).sort_values(['code', 'code_']).drop_duplicates('code').reset_index(drop=True)        
        print(f'{self.aligned.shape = }\n{self.aligned.head(self.aligned.shape[0])}')        
        return

    def _find_missalign(self):
        # find the 20 or so 2008 codes that are not in the 2020 codes and line them up by hand (in the "miss" table)
        miss = []
        for r1 in self.old.itertuples():
            found = False
            for r2 in self.aligned.itertuples():
                if r1.name_ in r2.name or r2.name in r1.name_:
                    print(f'FOUND {r1.name_ = } {r2.name = }')
                    found = True
                    break
            if not found:
                miss.append([r1.type_, r1.code_, r1.name_])
        self.miss = pd.DataFrame(miss).astype(str)
        self.miss.columns = ['type_', 'code_', 'name_']
        with pd.ExcelWriter('../data/BASIC_FOR_DATA/working.xlsx') as writer:
            for f in ['old', 'new', 'miss', 'aligned']:
                df = eval(f'self.{f}')
                df.to_excel(writer, sheet_name=f, index=False)
                print(f'saving {f} {df.shape = }\n{df.head()}')
        return

In [32]:
class SubfieldGroupFromNames(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_all_topics(self):
        self.topics = pd.read_excel('../data/BASIC_TOPIC_DATA/OA_topic_table.xlsx')
        print(f'{self.topics.shape = }\n{self.topics.head()}')
        self._field()
        self._subfield()
        self._extract_fors()
        self._process_subfield_to_group()
        self._load_subfield_to_group()
        return
    
    def _extract_fors(self):
        self.fors = self.db.sql("SELECT * FROM arc.for").df()
        print(f'{self.fors.shape = }\n{self.fors.head()}')
        self.divisions = self.fors[self.fors.type == 'DIVISION'][['code', 'name']].sort_values('code').reset_index(drop=True)
        print(f'{self.divisions.shape = }\n{self.divisions.head()}')
        self.groups = self.fors[self.fors.type == 'GROUP'][['code', 'name']].sort_values('code').reset_index(drop=True)
        print(f'{self.groups.shape = }\n{self.groups.head()}')
        return

    def _field(self):
        self.fields = self.topics.sort_values('field_id')[['field_id', 'field_name']].drop_duplicates('field_id').reset_index(drop=True)
        print(f'{self.fields.shape = }\n{self.fields.head()}')
        return

    def _subfield(self):
        self.subfields = self.topics.sort_values('subfield_id')[['subfield_id', 'subfield_name']].drop_duplicates('subfield_id').reset_index(drop=True)
        for col in ['group_code', 'group_name']:
            self.subfields[col] = pd.NA
        print(f'{self.subfields.shape = }\n{self.subfields.head()}\n{self.subfields.iloc[:, 2:4].head()}')
        return
    
    @staticmethod
    def _similar_names(self, name1=None, name2=None):
        s1 = set(name1.lower().replace(',', '').replace('and ', '').split(' '))
        s2 = set(name2.lower().replace(',', '').replace('-', '').replace('and ', '').split(' '))
        jacard = len(s1.intersection(s2))/len(s1.union(s2))
        if jacard > 0.35:
            print(f'{jacard = } {name1 = } {name2 = } {s1 = } {s2 = }')
            return True
        return False

    def _process_subfield_to_group(self):
        i, j = 1, 1
        for subfield in self.subfields.itertuples():
            found = False
            for group in self.groups.itertuples():
                if group.name in subfield.subfield_name or subfield.subfield_name in group.name:
                    # print(f'FOUND {i = } {subfield.Index = } {len(self.subfields) = } {subfield.subfield_name = }')
                    i += 1
                    found = True
                    self.subfields.iloc[subfield.Index, 2:4] = [group.code, group.name] 
                    break
                else:
                    if test := self._similar_names(group.name, subfield.subfield_name):
                        # print(f'FOUND {i = } {subfield.Index = } {len(self.subfields) = } {subfield.subfield_name = }')
                        i += 1
                        found = True
                        self.subfields.iloc[subfield.Index, 2:4] = [group.code, group.name] 
                        break
            if not found:
                print(f'MISSED {j = } {subfield.subfield_name = }')
                j += 1
        with pd.ExcelWriter('../data/BASIC_TOPIC_DATA/topic_to_for_working.xlsx') as writer:
            self.groups.to_excel(writer, sheet_name='groups', index=False)
            self.subfields.to_excel(writer, sheet_name='subfields', index=False)
        return
    
    def _load_subfield_to_group(self):
        subfield_to_group = pd.read_excel('../data/BASIC_TOPIC_DATA/subfields_groups_SAVE.xlsx', sheet_name='subfields_groups', dtype=str).iloc[:,0:4]
        print(f'{subfield_to_group.shape = }\n{subfield_to_group.head()}')
        self.db.sql("CREATE OR REPLACE TABLE arc.subfields_groups AS SELECT * FROM subfield_to_group")
        self.db.sql("CREATE OR REPLACE TABLE ERA.subfields_groups AS SELECT * FROM subfield_to_group")        
        self.db.sql("SELECT * FROM arc.subfields_groups").show()
        return

In [33]:
class SubfieldGroupFromJournals(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_ERA_data(self):
        sql = """
                SELECT sub.for_code,
                        f.for_description AS for_name,
                        replace(UNNEST(topic_share).subfield.id, 'https://openalex.org/subfields/', '') AS subfield_code,
                        UNNEST(topic_share).subfield.display_name AS subfield_name,
                        UNNEST(topic_share).value AS topic_score
                    FROM
                        (
                            SELECT unnest(m.for_codes) AS for_code,
                                s.topic_share 
                            FROM ERA.matched_sources m
                                LEFT JOIN ERA.sources_all_oa s
                                    ON m.source_id = s.source_id
                        ) sub
                    LEFT JOIN ERA.for_codes f
                        ON sub.for_code = f.for_code
                    WHERE length(sub.for_code) = 4
            """
        self.cross = self.db.sql(sql).df()
        print(f'CROSS {self.cross.shape = }\n{self.cross.head()}')
        self.summary = self.cross.groupby(['subfield_code', 'subfield_name', 'for_code', 'for_name', ], as_index=False).\
            topic_score.sum().sort_values('topic_score', ascending=False).drop_duplicates(subset=['subfield_code']).sort_values('subfield_code')
        with pd.ExcelWriter('../data/BASIC_TOPIC_DATA/topic_to_for_by__ejl.xlsx') as writer:
            self.summary.to_excel(writer, sheet_name='subfield_group', index=False)
        self._load_subfield_to_group()
        print(f'SUMMARY {self.summary.shape = }\n{self.summary.head()}')
        return
    
    def _load_subfield_to_group(self):
        subfield_to_group = pd.read_excel('../data/BASIC_TOPIC_DATA/topic_to_for_by__ejl.xlsx', sheet_name='subfield_group', dtype=str).iloc[:,0:4]
        print(f'{subfield_to_group.shape = }\n{subfield_to_group.head()}')
        self.db.sql("CREATE OR REPLACE TABLE arc.subfields_groups_ejl AS SELECT * FROM subfield_to_group")
        self.db.sql("CREATE OR REPLACE TABLE ERA.subfields_groups_ejl AS SELECT * FROM subfield_to_group")        
        self.db.sql("SELECT * FROM arc.subfields_groups").show()
        return

In [34]:
class ExtractORCID(SetUp):

    def __init__(self):
        super().__init__()
        directory_path = "/home/lc/s/orcid"
        return
    
    def extract_orcid(self, refresh=False):

        if refresh:
            json_file_path = "/home/lc/s/orcid/records.jsonl.gz"
            print(f'{json_file_path = } {Path(json_file_path).exists() = }')
            sql = f"""
                    CREATE OR REPLACE TABLE orcid.orcid_original_2023 AS
                        SELECT sub.*,
                                normalName.middle AS middle,
                                normalName.first || ' ' || normalName.last AS fullName_short,
                                IF(len(normalName.middle) == 0, 
                                    normalName.first || ' ' || normalName.last,
                                    normalName.first || ' ' || normalName.middle || ' ' || normalName.last) AS fullName,
                            FROM (SELECT *,
                                        normalise_name(name) AS normalName
                                    FROM read_json_auto('{json_file_path}', ignore_errors=True, sample_size=8056)) sub
                            LIMIT 4096
                """
            self.db.sql(sql)

        print("orcid.orcid_original_2023")
        self.db.sql("COPY orcid.orcid_original_2023 TO '../data/BASIC_ROR_DATA/orcid_original_20203.csv' (FORMAT csv, DELIMITER '|', HEADER)")
        self.db.sql("SELECT * FROM orcid.orcid_original_2023").show()
        self.db.sql("SELECT count(*), count(employments), count(works) FROM orcid.orcid_original_2023").show()
        return 
    
    def extract_ror(self, refresh=False):

        if refresh:
            in_file = '../data/BASIC_ROR_DATA/v1.63-2025-04-03-ror-data_schema_v2.csv'
            print(f'{Path(in_file).exists() = } {in_file = }')
            sql = """ 
                    SELECT *
                        FROM read_csv('../data/BASIC_ROR_DATA/v1.63-2025-04-03-ror-data_schema_v2.csv' )
                """
            df = self.db.sql(sql).df()[['id', 'locations.geonames_details.country_code','names.types.ror_display']]
            df.columns = ['ror_id', 'country_code', 'institution_name']
            df['ror_id'] = [id.replace('https://ror.org/', '') for id in df.ror_id]
            self.db.sql("CREATE OR REPLACE TABLE orcid.ror AS (SELECT * FROM df)")

        print("orcid.ror")
        self.db.sql("SELECT * FROM orcid.ror").show()
        self.db.sql("SELECT count(*) FROM orcid.ror").show()
        # self._get_ror_country()
        return
    
    def load_orcid_ror(self, refresh=False):

        if refresh:
            sql = """
                    CREATE OR REPLACE TABLE orcid.orcid_ror AS 
                        SELECT sub.orcid,
                                name,
                                normalName.first || ' ' || normalName.last AS fullName,
                                institution,
                                country_code,
                                country_code == 'AU' AS hasOZ 
                            FROM
                                (
                                SELECT DISTINCT 
                                    orcid,
                                    name,
                                    normalName, 
                                    unnest(employments).name AS institution,
                                    unnest(employments).xrefs.ror AS ror
                                FROM orcid.orcid_original_2023
                                WHERE employments NOT NULL
                                ORDER BY orcid
                                ) sub
                                LEFT JOIN orcid.ror
                                    ON sub.ror = ror_id
                                    WHERE ror NOT NULL
                            ORDER BY hasOZ DESC
                """
            self.db.sql(sql)

        print("orcid.orcid_ror")
        self.db.sql("SELECT * FROM orcid.orcid_ror").show()
        self.db.sql("SELECT count(*) FROM orcid.orcid_ror").show()
        return
    
    def reduce_orcid_ror(self, refresh=False):

        if refresh:
            self.db.sql("CREATE OR REPLACE TABLE memory.OZnames AS SELECT DISTINCT fullName FROM orcid.orcid_ror WHERE hasOZ is true")
            sql = """
                    CREATE OR REPLACE TABLE orcid.orcid_OZ AS
                        SELECT fullName,
                                list_distinct(list(hasOZ)) as hasOZs,
                                list_distinct(list(country_code)) AS country_codes,
                                list_distinct(list(orcid)) AS orcids,
                                list_distinct(list(institution)) AS institutions,
                                list_distinct(list(name)) AS names,
                            FROM orcid.orcid_ror
                            WHERE list_contains((SELECT list(DISTINCT fullName) FROM memory.OZnames), fullName) = true
                            GROUP BY fullName
                """
            self.db.sql(sql)

            sql = """
                    CREATE OR REPLACE TABLE orcid.orcid_notOZ AS
                        SELECT fullName,
                                list_distinct(list(hasOZ)) as hasOZs,
                                list_distinct(list(country_code)) AS country_codes,
                                list_distinct(list(orcid)) AS orcids,
                                list_distinct(list(institution)) AS institutions,
                                list_distinct(list(name)) AS names,
                            FROM orcid.orcid_ror
                            WHERE list_contains((SELECT list(DISTINCT fullName) FROM memory.OZnames), fullName) = false
                            GROUP BY fullName            
                """
            self.db.sql(sql)

        print("orcid.orcid_OZ")
        self.db.sql("SELECT * FROM orcid.orcid_OZ").show()
        self.db.sql("SELECT count(*) FROM orcid.orcid_OZ").show()

        print("orcid.orcid,notOZ")
        self.db.sql("SELECT * FROM orcid.orcid_notOZ").show()
        self.db.sql("SELECT count(*) FROM orcid.orcid_notOZ").show()
        return

In [35]:
def main():
    
    # lfor = FieldsOfResearch()
    # lfor.load_original_for_tables()

    # sgfn = SubfieldGroupFromNames()
    # sgfn.extract_all_topics()

    # sgfj = SubfieldGroupFromJournals()
    # sgfj.extract_ERA_data()

    eo = ExtractORCID()
    eo.extract_orcid(refresh=True)
    # eo.extract_ror(refresh=False)
    # eo.load_orcid_ror(refresh=False)
    # eo.reduce_orcid_ror(refresh=True)

In [36]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────────┬─────────┬──────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────